In [1]:
import numpy as np
import pandas as pd
import psycopg as pg
import mlflow
import os
from catboost import CatBoostClassifier
from mlxtend.feature_selection import SequentialFeatureSelector as SFS
from mlxtend.plotting import plot_sequential_feature_selection as plot_sfs
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import matplotlib.pyplot as plt

In [42]:
TABLE_NAME = "clean_users_churn"

TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000

EXPERIMENT_NAME = "Explore_search_hyperparams"
RUN_NAME = 'model_search'
REGISTRY_MODEL_NAME = "Applied_hyperparams"

In [3]:
configs = {"sslmode": "require", "target_session_attrs": "read-write"}
db_credits = {
    "host": os.getenv("DB_DESTINATION_HOST"),
    "port": os.getenv("DB_DESTINATION_PORT"),
    "dbname": os.getenv("DB_DESTINATION_NAME"),
    "user": os.getenv("DB_DESTINATION_USER"),
    "password": os.getenv("DB_DESTINATION_PASSWORD")
}

configs.update(db_credits)

In [4]:
with pg.connect(**configs) as conn:
    with conn.cursor() as cur:
        cur.execute(f"SELECT * FROM {TABLE_NAME}")

        data = cur.fetchall()

        columns = [col.name for col in cur.description]

In [5]:
df = pd.DataFrame(data=data, columns=columns)

In [6]:
features = ["monthly_charges", "total_charges", "senior_citizen"]
target = "target"

In [7]:
df[features]

,monthly_charges,total_charges,senior_citizen
0,19.80,202.25,0
1,20.15,20.15,0
2,59.90,3505.10,0
3,59.60,2970.30,0
4,55.30,1530.60,0
...,...,...,...
7019,69.50,1652.10,0
7020,76.00,1588.75,0
7021,93.60,3366.05,0
7022,95.65,778.10,0


In [7]:
test_size = 0.2
split_column = features


In [8]:
df.sort_values(by=["begin_date"], inplace=True)


In [9]:
X_train, X_test, y_train, y_test = train_test_split(df[features], df[target], test_size=test_size, shuffle=False)


In [10]:
X_train

,monthly_charges,total_charges,senior_citizen
4367,92.45,6440.25,1
4462,117.80,8684.80,0
3317,104.15,7689.95,1
951,108.05,7532.15,0
6850,108.60,7690.90,0
...,...,...,...
4006,89.00,605.45,1
2964,77.85,299.20,0
5719,34.20,256.60,0
3216,75.35,564.65,0


In [11]:
print(f"Размер выборки для обучения: {X_train.shape}")
print(f"Размер выборки для теста: {X_test.shape}")

Размер выборки для обучения: (5619, 3)
Размер выборки для теста: (1405, 3)


In [12]:
loss_function = "Logloss"
task_type = 'CPU'
random_seed = 0
iterations = 300
verbose = False

In [ ]:
param_grid = {
    'iterations': [50, 100, 300, 500],
    'learning_rate': [0.01, 0.1, 0.3],
    'loss_function': ["Logloss", "MultiClass"],
    'depth': [4, 6, 8, 10],
    'min_data_in_leaf': [50, 100, 200],
}

In [19]:
params_distribution = {
    'iterations': [50, 100, 300, 500],
    'learning_rate': [0.01, 0.1, 0.3],
    'loss_function': ["Logloss", "MultiClass"],
    'depth': np.arange(1, 10),
    'min_data_in_leaf': np.arange(10, 200, 10),
}

In [20]:
model = CatBoostClassifier(random_seed=random_seed, task_type=task_type)

In [21]:
cv = RandomizedSearchCV(estimator=model, param_distributions=params_distribution, cv=2, scoring='roc_auc', n_jobs=-1)

In [24]:
cv = GridSearchCV(estimator=model, param_grid=param_grid, cv=2, scoring="roc_auc", n_jobs=-1)

In [22]:
clf = cv.fit(X_train, y_train)

0:	learn: 0.6426823	total: 72ms	remaining: 3.53s
0:	learn: 0.6600946	total: 72.1ms	remaining: 3.53s
1:	learn: 0.5980039	total: 81.9ms	remaining: 1.97s
1:	learn: 0.6105498	total: 80.9ms	remaining: 1.94s
2:	learn: 0.5596083	total: 90ms	remaining: 1.41s
3:	learn: 0.5295307	total: 91.8ms	remaining: 1.05s
2:	learn: 0.5695092	total: 85ms	remaining: 1.33s
4:	learn: 0.5059957	total: 93.3ms	remaining: 840ms
3:	learn: 0.5370124	total: 90.6ms	remaining: 1.04s
4:	learn: 0.5125520	total: 92.7ms	remaining: 834ms
5:	learn: 0.4821196	total: 106ms	remaining: 780ms
6:	learn: 0.4617777	total: 111ms	remaining: 681ms
5:	learn: 0.4875642	total: 101ms	remaining: 744ms
7:	learn: 0.4436756	total: 119ms	remaining: 622ms
6:	learn: 0.4657542	total: 106ms	remaining: 652ms
8:	learn: 0.4255513	total: 127ms	remaining: 576ms
9:	learn: 0.4125158	total: 128ms	remaining: 513ms
7:	learn: 0.4453083	total: 110ms	remaining: 580ms
10:	learn: 0.3997061	total: 134ms	remaining: 477ms
11:	learn: 0.3910318	total: 136ms	remaining: 

In [23]:
print("Лучшие гиперпараметры:", clf.best_params_)
print("Лучший счет:", clf.best_score_)

Лучшие гиперпараметры: {'min_data_in_leaf': 10, 'loss_function': 'MultiClass', 'learning_rate': 0.1, 'iterations': 100, 'depth': 3}
Лучший счет: 0.8032461936938639


In [24]:
best_model = clf.best_estimator_

In [25]:
best_model.fit(X_train, y_train)

0:	learn: 0.6603001	total: 1.73ms	remaining: 171ms
1:	learn: 0.6252010	total: 3.7ms	remaining: 181ms
2:	learn: 0.5961991	total: 5.41ms	remaining: 175ms
3:	learn: 0.5725272	total: 7.14ms	remaining: 171ms
4:	learn: 0.5518655	total: 9.15ms	remaining: 174ms
5:	learn: 0.5341208	total: 10.8ms	remaining: 169ms
6:	learn: 0.5196833	total: 12.5ms	remaining: 166ms
7:	learn: 0.5064808	total: 14.2ms	remaining: 164ms
8:	learn: 0.4953497	total: 15.8ms	remaining: 160ms
9:	learn: 0.4861007	total: 17.4ms	remaining: 157ms
10:	learn: 0.4779315	total: 19.1ms	remaining: 154ms
11:	learn: 0.4710818	total: 20.8ms	remaining: 153ms
12:	learn: 0.4645891	total: 22.6ms	remaining: 151ms
13:	learn: 0.4592762	total: 24.4ms	remaining: 150ms
14:	learn: 0.4541532	total: 26.1ms	remaining: 148ms
15:	learn: 0.4491152	total: 27.8ms	remaining: 146ms
16:	learn: 0.4453381	total: 29.5ms	remaining: 144ms
17:	learn: 0.4413528	total: 31.2ms	remaining: 142ms
18:	learn: 0.4394500	total: 32.9ms	remaining: 140ms
19:	learn: 0.4353733	to

In [26]:
test_score = best_model.score(X_test, y_test)

In [28]:
cv_results = pd.DataFrame(clf.cv_results_)

In [29]:
best_params = clf.best_params_

In [30]:
best_params

{'min_data_in_leaf': 10,
 'loss_function': 'MultiClass',
 'learning_rate': 0.1,
 'iterations': 100,
 'depth': 3}

In [38]:
model_best = CatBoostClassifier(**best_params, random_seed=random_seed, task_type=task_type)

In [31]:
prediction = best_model.predict(X_test)
probas = best_model.predict_proba(X_test)[:, 1]

In [32]:
metrics = {}

In [33]:
from sklearn.metrics import confusion_matrix, roc_auc_score, precision_score, recall_score, f1_score, log_loss

In [34]:
_, err1, _, err2 = confusion_matrix(y_test, prediction, normalize="all").ravel()
auc = roc_auc_score(y_test, probas)
precision = precision_score(y_test, prediction)
recall = recall_score(y_test, prediction)
f1 = f1_score(y_test, prediction)
logloss = log_loss(y_test, prediction)

In [35]:
metrics["err1"] = err1
metrics["err2"] = err2
metrics["auc"] = auc
metrics["precision"] = precision
metrics["recall"] = recall
metrics["f1"] = f1
metrics["logloss"] = logloss

In [36]:
metrics["mean_fit_time"] = cv_results["mean_fit_time"].mean()# среднее время обучения
metrics["std_fit_time"] =  cv_results["std_fit_time"].mean()# стандартное отклонение времени обучения
metrics["mean_test_score"] = cv_results["mean_test_score"].mean()# средний результат на тесте
metrics["std_test_score"] = cv_results["std_test_score"].mean()# стандартное отклонение результата на тесте
metrics["best_score"] = clf.best_score_

In [37]:
pip_requirements = "./requirements.txt"
signature = mlflow.models.infer_signature(X_test, prediction)
input_example = X_test[:10]

/home/mle-user/mle_projects/mlflow/.venv/lib/python3.10/site-packages/mlflow/models/signature.py:212: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  inputs = _infer_schema(model_input) if model_input is not None else None


In [38]:
os.environ["MLFLOW_S3_ENDPOINT_URL"] = "https://storage.yandexcloud.net"
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("AWS_SECRET_ACCESS_KEY")

In [39]:
mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")
mlflow.set_registry_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")

In [40]:
experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id

In [41]:
best_params

{'min_data_in_leaf': 10,
 'loss_function': 'MultiClass',
 'learning_rate': 0.1,
 'iterations': 100,
 'depth': 3}

In [44]:
REGISTRY_MODEL_NAME

'Applied_hyperparams'

In [45]:
with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    mlflow.log_metrics(metrics)
    mlflow.log_params(best_params)
    cv_info = mlflow.sklearn.log_model(cv, artifact_path="cv")
    model_info = mlflow.catboost.log_model(cb_model=best_model, artifact_path="models",
                                          registered_model_name=REGISTRY_MODEL_NAME,
                                          signature=signature,
                                           input_example=input_example,
                                           pip_requirements=pip_requirements)

Registered model 'Applied_hyperparams' already exists. Creating a new version of this model...
2026/01/23 02:52:01 INFO mlflow.tracking._model_registry.client: Waiting up to 300 seconds for model version to finish creation. Model name: Applied_hyperparams, version 2
Created version '2' of model 'Applied_hyperparams'.
